In [ ]:
# ! pip install nomad-lab
import h5py
import numpy as np
# np.seterr(over='raise')  # works nicely
# except FloatingPointError:
from pynxtools.nomad.parsers.parser import (
    _get_field,
    _get_field_stats_iuf_contiguous,
    _get_field_stats_iuf_chunked
)

n = 1_000_000
file_path = "welford.nxs"

In [ ]:
dtypes = [
    # int
    "i1",
    "i2",
    "i4",
    "i8",
    # uint
    "u1",
    "u2",
    "u4",
    "u8",
    # float
    "f2",
    "f4",
    "f8",
    "f16",
    # complex
    "c8",
    "c16",
    "c32"
]
prng = np.random.default_rng(seed=42)  # deterministic seeding
with h5py.File(file_path, "w") as h5w:
    # values = (2 * prng.random(size=n)) - 1  # (-1, +1)
    values = prng.random(size=n)  # [0, 1)
    for typ in dtypes:
        if typ.startswith("i"):
            dat = np.asarray(1000 * values, dtype=typ)
        else:
            dat = np.asarray(1000. * values, dtype=typ)
        h5w.create_dataset(f"{typ}_contiguous", data=dat)
        h5w.create_dataset(f"{typ}_chunked", data=dat, chunks=True)
        h5w.create_dataset(f"{typ}_deflate", data=dat, chunks=(100,), compression="gzip")

In [ ]:
with h5py.File(file_path, "r") as h5r:
    for typ in dtypes:
        for layout in ["chunked", "contiguous", "deflate"]:
            dataset_name = f"{typ}_{layout}"
            # retval = _get_field(h5r[dataset_name])
            # print(f"type {type(retval)}, shape {np.shape(retval)}")
            # del retval
            if typ.startswith("f"):
                typ_atol = np.finfo(typ).eps
            else:
                typ_atol = 1.0e-9
            print(f"{dataset_name}, {typ_atol}")
            if layout in ["chunked", "deflate"]:
               field_stats = _get_field_stats_iuf_chunked(h5r[dataset_name])
            elif layout in ["contiguous"]:
               field_stats = _get_field_stats_iuf_contiguous(h5r[dataset_name])
            # abs(a - b) <= atol + rtol * abs(b)
            if not np.isclose(field_stats['__mean'], np.mean(h5r[dataset_name][...]), rtol=0., atol=typ_atol) or not np.isclose(field_stats['__std'], np.std(h5r[dataset_name][...]), rtol=0., atol=typ_atol):
                print(f"!!!! {dataset_name} abs diff mean {np.abs(field_stats['__mean'] - np.mean(h5r[dataset_name][...]))}")
                print(f"!!!! {dataset_name} abs diff std {np.abs(field_stats['__std'] - np.std(h5r[dataset_name][...]))}")
            # else:
            #    print(dataset_name)
            # print(f"custom {field_stats['__mean']}, {field_stats['__std']}")
            # print(f"numpy {np.mean(h5r[dataset_name][...])}, {np.std(h5r[dataset_name][...])}")     